# Gnod — Scraping the Billboard Hot 100

**Goal.** Gnod wants to recommend songs that are *popular right now*, independent
of the user's taste. This notebook scrapes the current **Billboard Hot 100** —
the 100 most popular songs in the US this week — and saves it as
`billboard_hot100.csv`, the "popular songs" building block of the recommender.

Billboard is served as static HTML, so `requests` + `BeautifulSoup` are enough;
no browser automation is needed.

## Setup

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

## 1. Fetch the page

Billboard blocks requests that don't look like a browser (they get a `403`). We
avoid that by sending a browser **User-Agent** header. A status code of `200`
means the page came back correctly.

In [2]:
url = "https://www.billboard.com/charts/hot-100/"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.status_code

200

## 2. Find the chart rows

Each song sits in a `<ul class="o-chart-results-list-row">`. Inside a row:
- the **rank** is the first `<span class="c-label">`,
- the **title** is the `<h3 id="title-of-a-story">`,
- the **artist** is the `<span>` right after the title.

In [3]:
soup = BeautifulSoup(response.content, "html.parser")
rows = soup.select("ul.o-chart-results-list-row")
len(rows)

100

## 3. Scrape the whole chart

Loop over the 100 rows and pull out the rank, title and artist. For the artist we
pass `" "` as the separator to `get_text`: when a song has several artists, each
is its own `<a>` link, and the separator keeps them readable
(`Tame Impala & JENNIE` instead of `Tame Impala&JENNIE`). The rank is converted
to an integer so it sorts correctly.

In [4]:
rank = []
title = []
artist = []

for row in rows:
    title_tag = row.select_one("h3#title-of-a-story")
    rank.append(int(row.select_one("span.c-label").get_text(strip=True)))
    title.append(title_tag.get_text(strip=True))
    artist.append(title_tag.find_next_sibling("span").get_text(" ", strip=True))

print(len(rank), len(title), len(artist))

100 100 100


## 4. Build the DataFrame and check it

In [5]:
billboard_hot100 = pd.DataFrame({"rank": rank, "title": title, "artist": artist})

print(billboard_hot100.shape)
print(billboard_hot100.isna().sum())
billboard_hot100.head()

(100, 3)
rank      0
title     0
artist    0
dtype: int64


,rank,title,artist
0,1,Choosin' Texas,Ella Langley
1,2,Boston,Stella Lefty
2,3,Dracula,Tame Impala & JENNIE
3,4,Housewife,Tate McRae
4,5,Stupid Song,Olivia Rodrigo


## 5. Save for the rest of the project

In [6]:
billboard_hot100.to_csv("../data/billboard_hot100.csv", index=False)
print("saved billboard_hot100.csv")

saved billboard_hot100.csv


## A first popularity check

A quick first version of the Gnod idea: given a song, say whether it's in this
week's Hot 100, and if so suggest another popular song. The match ignores case
and extra spaces. In the full project the audio features of these songs are
collected (data-collection notebook) so the Streamlit app can show them in a
"Popular" tab.

In [7]:
def check_song(song):
    song_clean = song.strip().lower()
    titles_clean = billboard_hot100["title"].str.strip().str.lower()

    if song_clean in titles_clean.values:
        other_songs = billboard_hot100[titles_clean != song_clean]
        pick = other_songs.sample(1).iloc[0]
        print(f"Popular! You might also like: {pick['title']} by {pick['artist']}")
    else:
        print("Sorry, your song is not popular")


check_song(billboard_hot100["title"][0])
check_song("A song nobody has ever heard")

Popular! You might also like: Dying In This House by Mack Geiger
Sorry, your song is not popular


## Note

Chart sites redesign their pages from time to time. If `rows` comes back empty,
re-open the page with the browser's **Inspect** tool and update the CSS
selectors.